# SAC Collector Head-to-Head — Sprint 2 (seed=7): FQL × 4 tier sweep

**状态**：ACTIVE — sprint 2 启动决策 land 2026-05-25 (expert review 完成 commit `4edee37`)。Per-seed shard。

**目标**：执行 [`docs/arrival_v2_sac_collector_design.md`](../docs/arrival_v2_sac_collector_design.md) **rev.3 §4.0.8 v2 + §4.0.9** Sprint 2 — FQL（flow_steps=10, distill_alpha_bc=1.0）× 4 tier × **seed 7** = **4 run**（per-seed shard）。与 sprint 1 ReBRAC seed shard 1:1 对位组成 SAC-collector head-to-head 完整矩阵（algorithm × tier × seed）。

**Seed-split rationale**：原 12-run sweep 按 seed 拆成 3 个 notebook (seed=42 / 0 / 7)，每份 4 run / ~3.5h L4，可在 Colab Pro 多 session 并发开跑（理想 wall ~3.5h vs 串行 10-11h）。三份共同输出 12 run 与原协议完全等价。其它两份：[`_seed42`](./sac_collector_h2h_fql_sweep_seed42.ipynb) / [`_seed0`](./sac_collector_h2h_fql_sweep_seed0.ipynb) / [`_seed7`](./sac_collector_h2h_fql_sweep_seed7.ipynb)。

**前置 commits（已闭环）**：
- `97d394c` — SACCheckpointPolicy adapter (5 tests pass)
- `032b527` — Plan A 4 tier dataset collection completed
- `3cfd18a` — spec rev.3 §4.0.7 actual results land
- `5070862` — IPython shell magic refactor (sprint 1 style)
- `104c14c` — sprint 1 ReBRAC 24 run completed
- `4edee37` — sprint 1 expert-review verdict correction

**配置（spec §4.0.8 v2 锁定）**：

| 项 | 值 | 来源 |
|---|---|---|
| Algo | FQL（vanilla critic） | FQL P2 v1.4 standard |
| Flow-matching teacher | `--flow-steps 10 --flow-time-embed-dim 32 --teacher-lr 3e-4` | FQL P2 c1 alpha sweep frozen |
| Distill α_bc | `--distill-alpha-bc 1.0` | FQL P2 C-1 baseline (RESCUE-FAIL reference) |
| Datasets | 4 tier (random/medium/mexp/expert, step25k/425k/575k/600k) | Plan A 4 tier completed |
| Seed (本 shard) | **[7]** | 与 sprint 1 ReBRAC paired (full set = [42, 0, 7]) |
| Total steps | 200_000 (uniform sampling) — FQL P2 §6.2 sister | FQL P2 v1.4 |
| batch / hidden / layers | 256 / 256 / 3 | 同上 |
| lr / γ / τ | 3e-4 / 0.99 / 0.005 | 同上 |
| Manifest (eval) | `benchmarks/single_u10_cross_tgt15.json` (30 ep/run, test_seed=456) | 与 sprint 1 paired |

**预算 (本 shard, seed=7)**：

| 阶段 | run | wallclock |
|---|---:|---:|
| Train: FQL × 4 tier × 1 seed | 4 | ~3.3h L4 |
| evaluate_offline × 4 run | 4 | ~10 min L4 |
| **Sprint 2 (seed=7) total** | **4** | **~3.5h L4** |

→ 1 个 Colab Pro L4 session 紧凑可完。三份 (seed=42/0/7) 同时启动 → 理想总 wall ~3.5h。

**实测要点**（基于 sprint 1 expert review）：
1. **唯一可被 sprint 2 reliably falsify 的 finding 是 finding 2** (FQL vs ReBRAC β1=1 on SAC clean expert)。sprint 1 已知 SAC expert ceiling = 0.889 (ReBRAC β1∈{1,4} 都达到此值)。
2. **SAC mexp tier 是有 statistical power 的 cell** — sprint 1 显示 β1=1 stratified CI 不跨 0 (β1=1 wins). FQL on mexp 数字关键：是否 ≥ ReBRAC β1=1 的 0.800？
3. **SAC random tier degenerate** — FQL 几乎必然也 0%，不会带来 finding signal。
4. **SAC medium tier informative**: ReBRAC β1=1 0.656 vs β1=4 0.633 (direction-only, CI 跨 0)。FQL 数字定 finding 1 cross-algo generalization。

**Pre-registered finding 2 + 加强 narrative**（详 spec §4.0.9）：
- **Finding 2 verify**: FQL on SAC expert ↔ ReBRAC β1=1 paired Δ CI 不跨 0 + Δ < 0 → 复现 FQL P2 C-1 RESCUE-FAIL on SAC clean regime; CI 跨 0 / Δ ≥ 0 → SAC dataset-bound regime 允许 FQL 也达到 ceiling
- **Cross-algorithm matrix complete**: ReBRAC β1∈{1,4} + FQL 全 12 cell × 4 tier × 3 seed (sprint 1 + sprint 2 共 36 run)

## 0. 环境检查

In [2]:
import torch
print(f"PyTorch:        {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device name:    {torch.cuda.get_device_name(0)}")

PyTorch:        2.10.0+cu128
CUDA available: True
Device name:    NVIDIA L4


## 1. 挂载 Drive + cd 到项目根

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
%cd drive/"MyDrive"/"Colab Notebooks"/"new_offRL"/"rl_v2_5"
!pwd

/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5
/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5


## 2. Sanity check — 4 dataset + manifest 存在

**预期**：5 项 ✅。任何 ⚠️ 都需停下定位（不要跳过 sanity 直接开跑）。

In [5]:
from pathlib import Path

TIERS = [
    ("random", "step25k"),
    ("medium", "step425k"),
    ("mexp",   "step575k"),
    ("expert", "step600k"),
]

def dataset_dir(tier: str, step_tag: str) -> Path:
    return Path(f"offline_data/sac_{tier}_s0_h4_arrival_v2_re150_u10cross_seed46_{step_tag}_ep1000")

MANIFEST = Path("benchmarks/single_u10_cross_tgt15.json")

all_ok = True
for tier, step_tag in TIERS:
    d = dataset_dir(tier, step_tag)
    npz = d / "transitions.npz"
    if npz.exists():
        print(f"  ✅ {tier:10s}  {npz}  ({npz.stat().st_size/1e6:.1f} MB)")
    else:
        print(f"  ⚠️ {tier:10s}  {npz}  NOT FOUND")
        all_ok = False

print()
if MANIFEST.exists():
    print(f"  ✅ manifest  {MANIFEST}  ({MANIFEST.stat().st_size/1e3:.1f} KB)")
else:
    print(f"  ⚠️ manifest  {MANIFEST}  NOT FOUND — 用 generate_standard_benchmarks 先生成")
    all_ok = False

print()
if all_ok:
    print("✅ Sanity check passed — 可以开始 training.")
else:
    raise FileNotFoundError("Sanity check failed — 见上面 ⚠️ 项，定位后 rerun.")

  ✅ random      offline_data/sac_random_s0_h4_arrival_v2_re150_u10cross_seed46_step25k_ep1000/transitions.npz  (21.4 MB)
  ✅ medium      offline_data/sac_medium_s0_h4_arrival_v2_re150_u10cross_seed46_step425k_ep1000/transitions.npz  (29.9 MB)
  ✅ mexp        offline_data/sac_mexp_s0_h4_arrival_v2_re150_u10cross_seed46_step575k_ep1000/transitions.npz  (20.6 MB)
  ✅ expert      offline_data/sac_expert_s0_h4_arrival_v2_re150_u10cross_seed46_step600k_ep1000/transitions.npz  (13.7 MB)

  ✅ manifest  benchmarks/single_u10_cross_tgt15.json  (17.0 KB)

✅ Sanity check passed — 可以开始 training.


## 3. Config matrix — 12 run plan

**Run identifier**：`fql__{tier}__seed_{42|0|7}` — 与 spec §4.0.8 v2 输出 schema 对齐。

**Skip-resume 协议**：per run 检查 `<save-dir>/agent_final.pt` 是否存在；存在则 skip（train 函数末尾才写，不会 false-positive）。

In [6]:
SEEDS = [7]  # per-seed shard; full set = [42, 0, 7] (seed42 / seed0 / seed7 notebooks)

CKPT_ROOT = Path("checkpoints/offline/sac_collector_h2h/fql")
RESULTS_ROOT = Path("results/offline/sac_collector_h2h/fql")

def save_dir(tier: str, seed: int) -> Path:
    return CKPT_ROOT / tier / f"seed_{seed}"

def test_result_path(tier: str, seed: int) -> Path:
    return RESULTS_ROOT / tier / f"seed_{seed}" / "test_result.json"

# Print plan + check skip-state
n_total = 0
n_done = 0
for tier, step_tag in TIERS:
    for seed in SEEDS:
        n_total += 1
        sd = save_dir(tier, seed)
        done_marker = sd / "agent_final.pt"
        status = "✅ DONE" if done_marker.exists() else "⏳ TODO"
        if done_marker.exists():
            n_done += 1
        print(f"  [{status}]  fql | {tier:10s} | seed={seed}  →  {sd}")
    print()

print(f"=== {n_done}/{n_total} runs already complete (skip-resume) ===")

  [⏳ TODO]  fql | random     | seed=7  →  checkpoints/offline/sac_collector_h2h/fql/random/seed_7

  [⏳ TODO]  fql | medium     | seed=7  →  checkpoints/offline/sac_collector_h2h/fql/medium/seed_7

  [⏳ TODO]  fql | mexp       | seed=7  →  checkpoints/offline/sac_collector_h2h/fql/mexp/seed_7

  [⏳ TODO]  fql | expert     | seed=7  →  checkpoints/offline/sac_collector_h2h/fql/expert/seed_7

=== 0/4 runs already complete (skip-resume) ===


## 4. Train — FQL × 4 tier × 3 seed = 12 run (~10h L4)

**协议**（spec §4.0.8 v2 + FQL P2 c1 alpha sweep frozen flags）：`--algo fql --flow-steps 10 --distill-alpha-bc 1.0 --teacher-lr 3e-4 --flow-time-embed-dim 32`。

**Per-run skip-resume**：若 `<save-dir>/agent_final.pt` 已存在则跳过。Colab session crash 后重新运行本 cell 自动接续。

**实时输出**：训练用 `!python ...` IPython shell magic，stdout 直接流式 → 每 1000 step 的 `actor_loss/critic_loss/teacher_loss` 实时滚动显示。

**注意**：12 run × 50 min ≈ 10h，单 Colab Pro L4 session 接近 12h limit。若 session 中断，重启再跑本 cell 会自动 skip 已完成 run。需缩短 wall 时建议按 sprint 1 同样的脚本 (`/tmp/split_sprint1_by_seed.py` 改 SEEDS → 3 份 by seed)，理想 3 session 并发 wall ~3.5h。

In [7]:
import time
from pathlib import Path

t_start = time.time()

for tier, step_tag in TIERS:
    for seed in SEEDS:
        sd = save_dir(tier, seed)
        sd_str = str(sd)
        npz = str(dataset_dir(tier, step_tag) / "transitions.npz")
        manifest_str = str(MANIFEST)
        if (sd / "agent_final.pt").exists():
            print(f"[skip-train] fql | {tier} | seed={seed} (agent_final.pt exists)")
            continue
        sd.mkdir(parents=True, exist_ok=True)
        print(f"\n{'=' * 72}")
        print(f"  train fql | {tier} | seed={seed}  →  {sd}")
        print(f"{'=' * 72}")
        t0 = time.time()
        !python -m scripts.train_offline \
            --algo fql \
            --offline-data '{npz}' \
            --manifest '{manifest_str}' \
            --probe-layout s0 \
            --history-length 4 \
            --task-geometry cross_stream \
            --target-speed 1.5 \
            --objective arrival_v2 \
            --sampling-mode uniform \
            --total-steps 200000 \
            --batch-size 256 \
            --hidden-dim 256 \
            --num-hidden-layers 3 \
            --actor-lr 3e-4 \
            --critic-lr 3e-4 \
            --gamma 0.99 \
            --tau 0.005 \
            --flow-steps 10 \
            --flow-time-embed-dim 32 \
            --teacher-lr 3e-4 \
            --distill-alpha-bc 1.0 \
            --grad-clip-norm 10.0 \
            --normalizer-eps 1e-3 \
            --eval-every 0 \
            --skip-final-eval \
            --log-every 1000 \
            --seed {seed} \
            --device cuda \
            --save-dir '{sd_str}'
        print(f"[train done] fql | {tier} | seed={seed} in {(time.time() - t0) / 60:.1f} min")

print(f"\n=== Sprint 2 FQL train complete, total = {(time.time() - t_start) / 60:.1f} min ===")


  train fql | random | seed=7  →  checkpoints/offline/sac_collector_h2h/fql/random/seed_7
[offline] algo=fql transitions=180067 obs_dim=48 action_dim=2 protocol=deployable tensor_replay=on eval_workers=1 sampling=uniform total_steps=200000
[train] step=1 q=-1.395 critic=464.722 actor=1.989 bc=0.989 lambda=0.717
[train] step=1000 q=-6.727 critic=475.169 actor=1.086 bc=0.086 lambda=0.149
[train] step=2000 q=-10.002 critic=58.953 actor=1.074 bc=0.074 lambda=0.100
[train] step=3000 q=-11.727 critic=177.447 actor=1.076 bc=0.076 lambda=0.085
[train] step=4000 q=-14.204 critic=159.058 actor=1.084 bc=0.084 lambda=0.070
[train] step=5000 q=-17.864 critic=405.618 actor=1.071 bc=0.071 lambda=0.056
[train] step=6000 q=-21.154 critic=552.571 actor=1.089 bc=0.089 lambda=0.047
[train] step=7000 q=-26.600 critic=195.573 actor=1.082 bc=0.082 lambda=0.038
[train] step=8000 q=-28.529 critic=4.494 actor=1.075 bc=0.075 lambda=0.035
[train] step=9000 q=-31.588 critic=96.290 actor=1.071 bc=0.071 lambda=0.03

## 5. Evaluate — 12 run × 100 ep test on `single_u10_cross_tgt15`

**Per-run skip-resume**：若 `<results-dir>/test_result.json` 已存在则跳过；若 `<save-dir>/agent_final.pt` 不存在则跳过并打 warn（train 未完）。

**预算**：12 × 2 min ≈ 25 min L4。

In [8]:
import time

t_start_eval = time.time()

for tier, _ in TIERS:
    for seed in SEEDS:
        sd = save_dir(tier, seed)
        sd_str = str(sd)
        out_path = test_result_path(tier, seed)
        out_path_str = str(out_path)
        manifest_str = str(MANIFEST)
        if out_path.exists():
            print(f"[skip-eval] fql | {tier} | seed={seed} (test_result.json exists)")
            continue
        if not (sd / "agent_final.pt").exists():
            print(f"[warn] fql | {tier} | seed={seed}  missing agent_final.pt — train not done?")
            continue
        out_path.parent.mkdir(parents=True, exist_ok=True)
        print(f"\n{'=' * 72}")
        print(f"  eval  fql | {tier} | seed={seed}  →  {out_path}")
        print(f"{'=' * 72}")
        t0 = time.time()
        !python -m scripts.evaluate_offline \
            --checkpoint '{sd_str}' \
            --manifest '{manifest_str}' \
            --episodes 100 \
            --seed 456 \
            --device cuda \
            --output-json '{out_path_str}'
        print(f"[eval done] fql | {tier} | seed={seed} in {(time.time() - t0) / 60:.1f} min")

print(f"\n=== Eval all complete, total = {(time.time() - t_start_eval) / 60:.1f} min ===")


  eval  fql | random | seed=7  →  results/offline/sac_collector_h2h/fql/random/seed_7/test_result.json
reward_objective      : arrival_v2
energy_cost_gain      : 0.000000
safety_cost_gain      : 0.000000
episodes              : 30
success_rate          : 0.0%
avg_return            : -338.57 +/- 57.15
avg_safety_cost       : 19.703 +/- 14.339
avg_time_s            : 86.51 +/- 38.93
avg_time_s_success    : nan
avg_energy            : 24245.21 +/- 13298.88
avg_path_length_m     : 86.70 +/- 38.63
avg_progress_ratio    : -0.993 +/- 0.811
avg_path_efficiency   : -0.400 +/- 0.245
termination           : {'out_of_bounds': 30}
benchmark_manifest    : benchmarks/single_u10_cross_tgt15.json
[eval done] fql | random | seed=7 in 1.3 min

  eval  fql | medium | seed=7  →  results/offline/sac_collector_h2h/fql/medium/seed_7/test_result.json
reward_objective      : arrival_v2
energy_cost_gain      : 0.000000
safety_cost_gain      : 0.000000
episodes              : 30
success_rate          : 63.3%
avg

## 6. Verify summary — 12 test_result.json → table

**输出**：
- per-tier mean ± std across 3 seeds (success_rate)
- (Optional) sprint 1 ReBRAC β1=1 / β1=4 per-tier paired comparison — 详细 paired bootstrap CI 留给本地 sprint 1+2 联合 analysis notebook

In [9]:
import json
import numpy as np

results: dict[str, list[float]] = {}
missing = []
for tier, _ in TIERS:
    seed_successes = []
    for seed in SEEDS:
        p = test_result_path(tier, seed)
        if not p.exists():
            missing.append((tier, seed))
            continue
        data = json.loads(p.read_text())
        sr = float(data.get("eval_success_rate", float("nan")))  # evaluate_offline.py outputs eval_success_rate in [0,1]
        seed_successes.append(sr)
    results[tier] = seed_successes

print("=== FQL head-to-head on SAC tier datasets (test 100 ep × 3 seed) ===\n")
print(f"{'tier':12s} {'FQL α_bc=1 (mean ± std)':25s} {'seeds':>20s}")
print("-" * 70)
for tier, _ in TIERS:
    s = results.get(tier, [])
    m = float(np.mean(s)) if s else float("nan")
    sd_ = float(np.std(s, ddof=0)) if len(s) > 1 else 0.0
    seed_repr = "  ".join(f"{v:.3f}" for v in s) if s else "(empty)"
    print(f"{tier:12s} {m:.3f} ± {sd_:.3f}{'':>13} {seed_repr:>20s}")

print()
if missing:
    print(f"⚠️ {len(missing)} runs missing test_result.json:")
    for m in missing:
        print(f"    {m}")
else:
    print("✅ All 12 runs have test_result.json — sprint 2 complete.")

=== FQL head-to-head on SAC tier datasets (test 100 ep × 3 seed) ===

tier         FQL α_bc=1 (mean ± std)                  seeds
----------------------------------------------------------------------
random       0.000 ± 0.000                             0.000
medium       0.633 ± 0.000                             0.633
mexp         0.967 ± 0.000                             0.967
expert       0.900 ± 0.000                             0.900

✅ All 12 runs have test_result.json — sprint 2 complete.


## 7. Next steps（sprint 2 完成后）

1. **Paired bootstrap analysis（本地 + sprint 1）**：合并 sprint 1 (24 ReBRAC) + sprint 2 (12 FQL) = 36 run 的 test_result.json，对 (tier × algo) 矩阵跑 paired bootstrap 1000 resample CI。重点对位：
   - expert ↔ FQL P2 E-uni（FQL vs ReBRAC β1=1 paired CI of Δ）
   - mexp ↔ FQL P2 M-uni-noise
   - medium ↔ FQL P2 M-multi-mix
2. **Finding 候选 2 判定**：在 expert tier, FQL distill_α_bc=1.0 vs ReBRAC β1=1 paired Δ 的 95% CI 若跨 0 → 复现 FQL P2 C-1 RESCUE-FAIL；CI 显著 < 0 → FQL 在 SAC clean expert 上更差（强化 paper 1 结论）。
3. **结果回收**：本 notebook 跑完结果落 `_completed.ipynb` 副本 commit 进 git 作实验记录（参考 audit / collection completed 模式）。